# Flight duration model: Adding departure time

In the previous exercise the departure time was bucketed and converted to dummy variables. Now you're going to include those dummy variables in a regression model for flight duration.

The data are in `flights`. The `km`, `org_dummy` and `depart_dummy` columns have been assembled into `features`, where `km` is index 0, `org_dummy` runs from index 1 to 7 and `depart_dummy` from index 8 to 14.

The data have been split into training and testing sets and a linear regression model, `regression`, has been built on the training data. Predictions have been made on the testing data and are available as `predictions`.

## Instructions

- Find the RMSE for predictions on the testing data.
- Find the average time spent on the ground for flights departing from OGG between 21:00 and 24:00.
- Find the average time spent on the ground for flights departing from OGG between 03:00 and 06:00.
- Find the average time spent on the ground for flights departing from JFK between 03:00 and 06:00.


In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()

In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [17]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M3-Regression/3_BucketingAndEngg/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0)).drop('mile')

from pyspark.ml.feature import StringIndexer

flights = StringIndexer(inputCol='org', outputCol='org_idx').fit(flights).transform(flights)

from pyspark.ml.feature import Bucketizer, OneHotEncoder, OneHotEncoderEstimator

# onehot = OneHotEncoder(inputCols=['org_idx'], outputCols=['org_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['org_idx'], outputCols=['org_dummy'])
flights = onehot.fit(flights).transform(flights)
print("--------------------------------------------------")
print(flights.printSchema())
flights.select('org','org_idx', 'org_dummy').groupBy('org').count().orderBy('count', ascending = False).show()
flights.select('org','org_idx', 'org_dummy').distinct().orderBy('org_idx').show()
print("--------------------------------------------------")
buckets = Bucketizer(splits=[0, 3, 6, 9, 12, 15, 18, 21, 24], inputCol='depart', outputCol='depart_bucket')
flights = buckets.transform(flights)
# onehot = OneHotEncoder(inputCols=['depart_bucket'], outputCols=['depart_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['depart_bucket'], outputCols=['depart_dummy'])
flights = onehot.fit(flights).transform(flights)

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['km', 'org_dummy', 'depart_dummy'], outputCol='features')
flights = assembler.transform(flights)
flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=13)
from pyspark.ml.regression import LinearRegression
regression = LinearRegression(labelCol='duration')
regression = regression.fit(flights_train)

print("""\nFeature columns:\n\n 0 — km\n 1 — ORD\n 2 — SFO\n 3 — JFK\n 4 — LGA\n 5 — SJC\n 6 — SMF\n 7 — TUS\n\
8 — 00:00 to 03:00\n9 — 03:00 to 06:00\n10 — 06:00 to 09:00\n11 — 09:00 to 12:00\n12 — 12:00 to 15:00\n\
13 — 15:00 to 18:00\n14 — 18:00 to 21:00\n""")
predictions = regression.transform(flights_test)


--------------------------------------------------
root
 |-- mon: integer (nullable = true)
 |-- dom: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- org: string (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- km: double (nullable = true)
 |-- org_idx: double (nullable = false)
 |-- org_dummy: vector (nullable = true)

None
+---+-----+
|org|count|
+---+-----+
|ORD|19337|
|SFO| 9557|
|JFK| 7958|
|LGA| 4995|
|SJC| 3057|
|SMF| 3032|
|TUS| 1055|
|OGG| 1009|
+---+-----+

+---+-------+-------------+
|org|org_idx|    org_dummy|
+---+-------+-------------+
|ORD|    0.0|(7,[0],[1.0])|
|SFO|    1.0|(7,[1],[1.0])|
|JFK|    2.0|(7,[2],[1.0])|
|LGA|    3.0|(7,[3],[1.0])|
|SJC|    4.0|(7,[4],[1.0])|
|SMF|    5.0|(7,[5],[1.0])|
|TUS|    6.0|(7,[6],[1.0])|
|OGG|    7.0|    (7,[],[])|
+---+-------+-------------+

---

In [ ]:
# Find the RMSE on testing data
from pyspark.ml.____ import ____
rmse = ____(____).____(____)
print("The test RMSE is", rmse)

# Average minutes on ground at OGG for flights departing between 21:00 and 24:00
avg_eve_ogg = regression.____
print(avg_eve_ogg)

# Average minutes on ground at OGG for flights departing between 03:00 and 06:00
avg_night_ogg = regression.____ + regression.____[9]
print(avg_night_ogg)

# Average minutes on ground at JFK for flights departing between 03:00 and 06:00
avg_night_jfk = regression.____ + regression.____[____] + regression.____[____]
print(avg_night_jfk)

In [5]:
print("Regression coefficients = {}".format(regression.coefficients))
# Find the RMSE on testing data
from pyspark.ml.evaluation import RegressionEvaluator
rmse = RegressionEvaluator(labelCol = 'duration').evaluate(predictions)
print("The test RMSE is", rmse)

# Average minutes on ground at OGG for flights departing between 21:00 and 24:00
avg_eve_ogg = regression.intercept
print(avg_eve_ogg)

# Average minutes on ground at OGG for flights departing between 03:00 and 06:00
avg_night_ogg = regression.intercept + regression.coefficients[9]
print(avg_night_ogg)

# Average minutes on ground at JFK for flights departing between 03:00 and 06:00
avg_night_jfk = regression.intercept + regression.coefficients[3] + regression.coefficients[9]
print(avg_night_jfk)

Regression coefficients = [0.07440555852185751,27.04304274796471,20.106334275083658,51.70583692128552,45.56822549928464,17.454798672975134,15.0032058475993,17.178642716704775,-14.600738737748829,1.7826941472610056,4.151717966388683,6.959275597279387,4.719898792106024,8.9250303798951,8.793700256036702]
The test RMSE is 10.812052722568886
10.475615792093903
12.258309939354909
63.964146860640426


Adding departure time resulted in a smaller RMSE. Nice!